Cell 1: Feature Grouping Dict. Generator

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

def preprocess_dataset(input_file):
    # load file based on extension
    if input_file.endswith('.csv'):
        df = pd.read_csv(input_file)
    elif input_file.endswith('.parquet'):
        df = pd.read_parquet(input_file)
    else:
        raise ValueError("Unsupported file format")

    # drop rows with nulls
    df = df.dropna()

    # drop duplicates
    df = df.drop_duplicates()

    # fill missing values with column mean
    for col in df.select_dtypes(include=['float64', 'int64']).columns:
        df.fillna(df.mean(numeric_only=True), inplace=True)

    # convert categorical columns to category dtype
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('category')

    return df

def normalize_features(df, feature_columns):
    # normalize specified feature columns
    scaler = StandardScaler()
    df[feature_columns] = scaler.fit_transform(df[feature_columns])
    return df, scaler # return scaler to inverse transform later if needed

def split_data(df, target_column, train_size=0.8, test_size=0.2, random_state=42):
    # split data into train and test sets
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=train_size, test_size=test_size, random_state=random_state, stratify=y)
    return X_train, X_test, y_train, y_test

if __name__ == "__main__":
    # example usage
    # input_file = 'datasets/Benign-Monday-no-metadata.parque'
    # target_column = 'target'
    # feature_columns = ['feature1', 'feature2', 'feature3']

    # df = preprocess_dataset(input_file)
    # df, scaler = normalize_features(df, feature_columns)
    # X_train, X_test, y_train, y_test = split_data(df, target_column)

    # print("Preprocessing complete.")
    # print(f"Training set size: {X_train.shape[0]}")
    # print(f"Test set size: {X_test.shape[0]}")


    def profile_column(series):
        s = series.dropna()
        dtype = s.dtype

        if pd.api.types.is_object_dtype(dtype) or pd.api.types.is_categorical_dtype(dtype):
            sample = s.astype(str).sample(min(10, len(s))).tolist()
            return {
                'dtype': dtype,
                'unique': s.nunique(),
                'sample_vals': sample,
                'is_numeric': False
            }

        sample = s.sample(min(10, len(s))).tolist()

        return {
            'min': s.min(),
            'max': s.max(),
            'mean': s.mean(),
            'std': s.std(),
            'unique': s.nunique(),
            'skew': s.skew(),
            'dtype': dtype,
            'sample_vals': sample,
            'is_numeric': True
        }


    def classify_column(col, series):

      name = col.lower()
      stats = profile_column(series)

      if not stats['is_numeric']:
          return "CATEGORICAL"

      if "flag" in name:
          return "BINARY"
      if "ratio" in name:
          return "RATIO"
      if any(k in name for k in ["bytes","packets","duration","iat","length","size"]):
          return "LOG_ROBUST"

      if stats["unique"] <= 3 and stats["min"] >= 0 and stats["max"] <= 1:
          return "BINARY"

      if 0 <= stats["min"] and stats["max"] <= 1.5:
          return "RATIO"

      if stats["skew"] > 2 or stats["max"] > 1000:
          return "LOG_ROBUST"

      if -50 < stats["min"] < 200 and stats["max"] < 300:
          return "SENSOR_CONTINUOUS"

      return "OTHER"

    # count number of features and list them
    cicids_dir = '/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Datasets/CICIDS2017'
    toniot_dir = '/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Datasets/TON-IoT'

    feature_registry = defaultdict(set)

    for file in os.listdir(cicids_dir):
        file_path = os.path.join(cicids_dir, file)
        df = preprocess_dataset(file_path)

        for col in df.columns:
            if col == 'target':
                continue

            group = classify_column(col, df[col])
            feature_registry[group].add(col)

    for dir in os.listdir(toniot_dir):
        file_path = os.path.join(toniot_dir, dir)

        for file in os.listdir(file_path):
            dataset_path = os.path.join(file_path, file)
            df = preprocess_dataset(dataset_path)

            for col in df.columns:
                if col == 'target':
                    continue

                group = classify_column(col, df[col])
                feature_registry[group].add(col)

    for group, cols in feature_registry.items():
        print(f"{group} = [")
        for c in sorted(cols):
            print(f'    "{c}",')
        print("]\n")


/tmp/ipython-input-1655276055.py:66: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(dtype) or pd.api.types.is_categorical_dtype(dtype):
/tmp/ipython-input-1655276055.py:66: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(dtype) or pd.api.types.is_categorical_dtype(dtype):
/tmp/ipython-input-1655276055.py:66: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(dtype) or pd.api.types.is_categorical_dtype(dtype):
/tmp/ipython-input-1655276055.py:66: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if p

SENSOR_CONTINUOUS = [
    "Protocol",
    "current_temperature",
    "fridge_temperature",
    "humidity",
    "pressure",
    "temperature",
]

RATIO = [
    "Down/Up Ratio",
    "Flow Duration",
    "duration",
]

LOG_ROBUST = [
    "Active Max",
    "Active Mean",
    "Active Min",
    "Active Std",
    "Avg Bwd Segment Size",
    "Avg Fwd Segment Size",
    "Avg Packet Size",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Header Length",
    "Bwd IAT Max",
    "Bwd IAT Mean",
    "Bwd IAT Min",
    "Bwd IAT Std",
    "Bwd IAT Total",
    "Bwd Packet Length Max",
    "Bwd Packet Length Mean",
    "Bwd Packet Length Min",
    "Bwd Packet Length Std",
    "Bwd Packets Length Total",
    "Bwd Packets/s",
    "FC1_Read_Input_Register",
    "FC2_Read_Discrete_Value",
    "FC3_Read_Holding_Register",
    "FC4_Read_Coil",
    "Flow Bytes/s",
    "Flow IAT Max",
    "Flow IAT Mean",
    "Flow IAT Min",
    "Flow IAT Std",
    "Flow Packets/s",
    "Fwd Act Data Packets",
   

/tmp/ipython-input-1655276055.py:66: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(dtype) or pd.api.types.is_categorical_dtype(dtype):
/tmp/ipython-input-1655276055.py:66: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(dtype) or pd.api.types.is_categorical_dtype(dtype):
/tmp/ipython-input-1655276055.py:66: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(dtype) or pd.api.types.is_categorical_dtype(dtype):


Cell 1.1: Import Right Dependencies and Initialize Directories

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

cicids_dir = '/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Datasets/CICIDS2017'
toniot_dir = '/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Datasets/TON-IoT'

Cell 2: Feature Grouping Dictionary

In [ ]:
SENSOR_CONTINUOUS = [
    "Protocol",
    "current_temperature",
    "fridge_temperature",
    "humidity",
    "pressure",
    "temperature",
]

RATIO = [
    "Down/Up Ratio",
    "Flow Duration",
    "duration",
]

LOG_ROBUST = [
    "Active Max",
    "Active Mean",
    "Active Min",
    "Active Std",
    "Avg Bwd Segment Size",
    "Avg Fwd Segment Size",
    "Avg Packet Size",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Header Length",
    "Bwd IAT Max",
    "Bwd IAT Mean",
    "Bwd IAT Min",
    "Bwd IAT Std",
    "Bwd IAT Total",
    "Bwd Packet Length Max",
    "Bwd Packet Length Mean",
    "Bwd Packet Length Min",
    "Bwd Packet Length Std",
    "Bwd Packets Length Total",
    "Bwd Packets/s",
    "FC1_Read_Input_Register",
    "FC2_Read_Discrete_Value",
    "FC3_Read_Holding_Register",
    "FC4_Read_Coil",
    "Flow Bytes/s",
    "Flow IAT Max",
    "Flow IAT Mean",
    "Flow IAT Min",
    "Flow IAT Std",
    "Flow Packets/s",
    "Fwd Act Data Packets",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Header Length",
    "Fwd IAT Max",
    "Fwd IAT Mean",
    "Fwd IAT Min",
    "Fwd IAT Std",
    "Fwd IAT Total",
    "Fwd Packet Length Max",
    "Fwd Packet Length Mean",
    "Fwd Packet Length Min",
    "Fwd Packet Length Std",
    "Fwd Packets Length Total",
    "Fwd Packets/s",
    "Fwd Seg Size Min",
    "Idle Max",
    "Idle Mean",
    "Idle Min",
    "Idle Std",
    "Init Bwd Win Bytes",
    "Init Fwd Win Bytes",
    "Packet Length Max",
    "Packet Length Mean",
    "Packet Length Min",
    "Packet Length Std",
    "Packet Length Variance",
    "Protocol",
    "Subflow Bwd Bytes",
    "Subflow Bwd Packets",
    "Subflow Fwd Bytes",
    "Subflow Fwd Packets",
    "Total Backward Packets",
    "Total Fwd Packets",
    "dns_qclass",
    "dns_qtype",
    "dns_rcode",
    "dst_bytes",
    "dst_ip_bytes",
    "dst_pkts",
    "dst_port",
    "http_request_body_len",
    "http_response_body_len",
    "http_status_code",
    "latitude",
    "longitude",
    "missed_bytes",
    "src_bytes",
    "src_ip_bytes",
    "src_pkts",
    "src_port",
]

BINARY = [
    "ACK Flag Count",
    "Bwd Avg Bulk Rate",
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "CWE Flag Count",
    "ECE Flag Count",
    "FIN Flag Count",
    "Fwd Avg Bulk Rate",
    "Fwd PSH Flags",
    "Fwd URG Flags",
    "PSH Flag Count",
    "RST Flag Count",
    "SYN Flag Count",
    "URG Flag Count",
    "label",
    "motion_status",
    "thermostat_status",
]

CATEGORICAL = [
    "Label",
    "conn_state",
    "date",
    "dns_AA",
    "dns_RA",
    "dns_RD",
    "dns_query",
    "dns_rejected",
    "door_state",
    "dst_ip",
    "http_method",
    "http_orig_mime_types",
    "http_resp_mime_types",
    "http_trans_depth",
    "http_uri",
    "http_user_agent",
    "http_version",
    "light_status",
    "proto",
    "service",
    "sphone_signal",
    "src_ip",
    "ssl_cipher",
    "ssl_established",
    "ssl_issuer",
    "ssl_resumed",
    "ssl_subject",
    "ssl_version",
    "temp_condition",
    "time",
    "type",
    "weird_addl",
    "weird_name",
    "weird_notice",
]

Cell 3: Load/Preprocess CICIDS2017 Dataset Helper Function

In [ ]:
def load_all_cicids():
    dfs = []
    for file in os.listdir(cicids_dir):
        if file.endswith('.parquet'):
            path = os.path.join(cicids_dir, file)
            df = load_raw_file(path)
            df = preprocess_dataframe(df)
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


Cell 4: Load/Preprocess TON_IoT Network Dataset Helper Function

In [ ]:
def load_ton_network():
  dfs = []

  for root, _, files in os.walk(toniot_dir):
    for f in files:
      if f.endswith('network.csv'):
        path = os.path.join(root, f)
        df = load_raw_file(path)
        df = preprocess_dataframe(df)
        dfs.append(df)

  return pd.concat(dfs, ignore_index=True)

Cell 5: Fit and Save Robust Network Profiles

In [ ]:
from sklearn.preprocessing import RobustScaler
import joblib
import os

os.makedirs("profiles", exist_ok=True)

df_cicids = load_all_cicids()
df_ton = load_ton_network()

def safe_log_transform(df, cols):
    for c in cols:
        col = df[c].values
        clip_val = np.nanpercentile(col, 99.9)

        col = np.clip(col, 0, clip_val)
        df[c] = np.log1p(col)

    return df

def select_existing(df, cols):
    return df[[c for c in cols if c in df.columns]]

df_all = pd.concat([
    select_existing(df_cicids, LOG_ROBUST),
    select_existing(df_ton, LOG_ROBUST)
], ignore_index=True)

df_all = safe_log_transform(df_all, df_all.columns)

scaler = RobustScaler().fit(df_all)

joblib.dump({
    "scaler": scaler,
    "columns": df_all.columns.tolist()
}, "profiles/network_log_robust_scaler.pkl")

/tmp/ipython-input-3288542634.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/tmp/ipython-input-3288542634.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/tmp/ipython-input-3288542634.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/tmp/ipython-input-3288542634.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/tmp/ipython-inp

['profiles/network_log_robust_scaler.pkl']

Cell 6: Normalize Network Data Helper Function

In [ ]:
def normalize_network(df):
    bundle = joblib.load("profiles/network_log_robust_scaler.pkl")
    scaler = bundle["scaler"]
    fit_cols = bundle["columns"]   # exact order from fit time

    # add any missing columns
    for col in fit_cols:
        if col not in df.columns:
            df[col] = 0

    # enforce same column order
    df_fit = df[fit_cols].copy()

    # log transform safely
    df_fit = safe_log_transform(df_fit, fit_cols)

    # scale
    df_fit = scaler.transform(df_fit)

    # put back into dataframe
    df[fit_cols] = df_fit

    return df


Cell 7: Normalize Sensor Data Helper Function

In [ ]:
def normalize_sensor(df):
    for col in SENSOR_CONTINUOUS:
        if col not in df:
            continue

        if "temp" in col:
            df[col] = (df[col] - 20) / 15
        elif "humidity" in col:
            df[col] = df[col] / 100
        elif "pressure" in col:
            df[col] = (df[col] - 1013) / 200

    return df


Cell 8: Normalize Network and Sensor Data Helper Function

In [ ]:
def normalize(df, source):
    if source in ["cicids", "ton_network"]:
        return normalize_network(df)
    elif source == "ton_sensor":
        return normalize_sensor(df)


Cell 9: Make Splits Helper Function

In [ ]:
from sklearn.model_selection import train_test_split

def make_splits(df, label_col="target"):

    X = df.drop(columns=[label_col])
    y = df[label_col]

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


Cell 10: Preprocess Dataframe Helper Function

In [ ]:
def preprocess_dataframe(df):
  if "Label" in df.columns:
        df.rename(columns={"Label": "target"}, inplace=True)

  drop_cols = ["src_ip", "dst_ip", "date", "time"]
  for col in drop_cols:
      if col in df.columns:
          df.drop(columns=[col], inplace=True)


  df = df.apply(pd.to_numeric, errors='ignore')
  df.replace([np.inf, -np.inf], np.nan, inplace=True)

  num_cols = df.select_dtypes(include=[np.number]).columns
  cat_cols = df.select_dtypes(include=["category", "object"]).columns

  df[num_cols] = df[num_cols].fillna(0)

  for c in cat_cols:
        if isinstance(df[c].dtype, pd.CategoricalDtype):
            df[c] = df[c].cat.add_categories(["unknown"])
            df[c] = df[c].fillna("unknown")
        else:
            df[c] = df[c].fillna("unknown")

  return df

Cell 11: Load Raw File Helper Function

In [ ]:
def load_raw_file(path):
  if path.endswith('.parquet'):
    return pd.read_parquet(path)
  if path.endswith('.csv'):
    return pd.read_csv(path)

  raise ValueError(f"Unsupported file type: {path}")

Cell 12: Load Cleaned Dataset Helper Function

In [ ]:
def load_dataset(path, source):
  df = load_raw_file(path)
  df = preprocess_dataframe(df)
  df = normalize(df, source)
  return make_splits(df)

Cell 13: Visualize Feature Space Dimensions

In [ ]:
file_path = '/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Datasets/CICIDS2017/Benign-Monday-no-metadata.parquet'

X_train, X_val, X_test, y_train, y_val, y_test = load_dataset(
    file_path, source="cicids")

print(X_train.shape, X_val.shape, X_test.shape)

/tmp/ipython-input-3288542634.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')


(321181, 92) (68825, 92) (68825, 92)


Cell 14: RandomForest Training and Macro F1

In [ ]:
df_all = load_all_cicids()
df_all = normalize_network(df_all)

X_train, X_val, X_test, y_train, y_val, y_test = make_splits(df_all)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

model = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))
print("Macro F1:", f1_score(y_test, preds, average="macro"))



/tmp/ipython-input-3288542634.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/tmp/ipython-input-3288542634.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/tmp/ipython-input-3288542634.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/tmp/ipython-input-3288542634.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/tmp/ipython-inp

                            precision    recall  f1-score   support

                    Benign       1.00      1.00      1.00    296598
                       Bot       0.76      0.67      0.71       215
                      DDoS       1.00      1.00      1.00     19202
             DoS GoldenEye       0.99      1.00      1.00      1543
                  DoS Hulk       1.00      1.00      1.00     25927
          DoS Slowhttptest       0.94      0.99      0.97       784
             DoS slowloris       1.00      1.00      1.00       808
               FTP-Patator       1.00      1.00      1.00       890
                Heartbleed       1.00      1.00      1.00         1
              Infiltration       1.00      0.60      0.75         5
                  PortScan       0.94      0.91      0.93       294
               SSH-Patator       1.00      0.98      0.99       483
  Web Attack � Brute Force       0.73      0.77      0.75       221
Web Attack � Sql Injection       1.00      0.33

Cell 14.1: Global Numeric Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = X_train.select_dtypes(include=[np.number]).columns

scaler_global = StandardScaler().fit(X_train[num_cols])

X_train[num_cols] = scaler_global.transform(X_train[num_cols])
X_val[num_cols]   = scaler_global.transform(X_val[num_cols])
X_test[num_cols]  = scaler_global.transform(X_test[num_cols])


Cell 14.2: X_* Type Sanity Check

In [ ]:
print("Any object dtypes?", any(dt == "object" for dt in X_train.dtypes.astype(str)))
print("NaNs in X_train?", np.isnan(X_train.values).any())
print("Infs in X_train?", np.isinf(X_train.values).any())
print("X_train dtype:", X_train.values.dtype)

Any object dtypes? False
NaNs in X_train? False
Infs in X_train? False
X_train dtype: float64


Cell 14.3: Ensure Accurate Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y_train)

LabelEncoder()

Cell 14.4: Temporary Overfit Model

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# small subset
N = 20000
Xs = X_train.iloc[:N]
ys = y_train.iloc[:N]

X_t = torch.tensor(Xs.values, dtype=torch.float32)
y_t = torch.tensor(le.transform(ys), dtype=torch.long)

loader = DataLoader(TensorDataset(X_t, y_t), batch_size=1024, shuffle=True)

class MLP_Overfit(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes),
        )
    def forward(self, x): return self.net(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
m = MLP_Overfit(X_t.shape[1], len(le.classes_)).to(device)

opt = torch.optim.Adam(m.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()  # NO weights for this test

for epoch in range(10):
    m.train()
    correct = total = 0
    loss_sum = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = m(xb)
        loss = crit(out, yb)
        loss.backward()
        opt.step()
        loss_sum += loss.item()

        pred = out.argmax(1)
        correct += (pred == yb).sum().item()
        total += yb.numel()

    print(f"Epoch {epoch+1} | loss {loss_sum/len(loader):.3f} | train acc {correct/total:.3f}")

Epoch 1 | loss 1.290 | train acc 0.805
Epoch 2 | loss 0.332 | train acc 0.920
Epoch 3 | loss 0.211 | train acc 0.960
Epoch 4 | loss 0.148 | train acc 0.972
Epoch 5 | loss 0.118 | train acc 0.975
Epoch 6 | loss 0.101 | train acc 0.976
Epoch 7 | loss 0.091 | train acc 0.977
Epoch 8 | loss 0.082 | train acc 0.979
Epoch 9 | loss 0.073 | train acc 0.981
Epoch 10 | loss 0.066 | train acc 0.983


Cell 14.5: Verify Label Encoding Consistency

In [ ]:
print("Classes:", list(le.classes_))
print("Example mapping:", {cls:i for i,cls in enumerate(le.classes_)})
print("Train unique codes:", np.unique(le.transform(y_train))[:20])
print("Test unique codes:", np.unique(le.transform(y_test))[:20])

Classes: ['Benign', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack � Brute Force', 'Web Attack � Sql Injection', 'Web Attack � XSS']
Example mapping: {'Benign': 0, 'Bot': 1, 'DDoS': 2, 'DoS GoldenEye': 3, 'DoS Hulk': 4, 'DoS Slowhttptest': 5, 'DoS slowloris': 6, 'FTP-Patator': 7, 'Heartbleed': 8, 'Infiltration': 9, 'PortScan': 10, 'SSH-Patator': 11, 'Web Attack � Brute Force': 12, 'Web Attack � Sql Injection': 13, 'Web Attack � XSS': 14}
Train unique codes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
Test unique codes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]


Cell 15: Convert X and y Values into Tensors

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def to_torch(X, y, le):
    y_enc = le.transform(y)
    X_t = torch.tensor(X.values, dtype=torch.float32)
    y_t = torch.tensor(y_enc, dtype=torch.long)
    return X_t, y_t

Xtr, ytr = to_torch(X_train, y_train, le)
Xva, yva = to_torch(X_val, y_val, le)
Xte, yte = to_torch(X_test, y_test, le)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=2048, shuffle=True)
val_loader = DataLoader(TensorDataset(Xva, yva), batch_size = 4096)
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size = 4096)

n_features = Xtr.shape[1]
n_classes = len(torch.unique(ytr))

Cell 16: MLP Definition

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
  def __init__(self, in_dim, n_classes):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(in_dim, 256),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(128, n_classes)
    )

  def forward(self, x):
    return self.net(x)

Cell 17: Define MLP Training Loop and Weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MLP(n_features, n_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(loader, train=True):
  model.train() if train else model.eval()
  total_loss = 0

  for Xb, yb in loader:
    Xb, yb = Xb.to(device), yb.to(device)

    if train:
      optimizer.zero_grad()

    out = model(Xb)
    loss = criterion(out, yb)

    if train:
      loss.backward()
      optimizer.step()

    total_loss += loss.item()

  return total_loss / len(loader)

Cell 18: MLP Training Per Epoch Data

In [ ]:
for epoch in range(10):
  train_loss = run_epoch(train_loader, train=True)
  val_loss = run_epoch(val_loader, train=False)

  print(f"Epoch: {epoch+1}, Train: {train_loss:.4f}, Val: {val_loss:.4f}")

Epoch: 1, Train: 0.0988, Val: 0.0213
Epoch: 2, Train: 0.0214, Val: 0.0138
Epoch: 3, Train: 0.0158, Val: 0.0116
Epoch: 4, Train: 0.0137, Val: 0.0106
Epoch: 5, Train: 0.0125, Val: 0.0099
Epoch: 6, Train: 0.0117, Val: 0.0098
Epoch: 7, Train: 0.0111, Val: 0.0092
Epoch: 8, Train: 0.0107, Val: 0.0089
Epoch: 9, Train: 0.0102, Val: 0.0090
Epoch: 10, Train: 0.0100, Val: 0.0089


Cell 19: MLP Macro F1 Evaluation

In [ ]:
from sklearn.metrics import classification_report, f1_score

model.eval()
all_preds = []
all_true = []

with torch.no_grad():
  for Xb, yb in test_loader:
    Xb = Xb.to(device)
    out = model(Xb)
    pred = torch.argmax(out, dim=1).cpu().numpy()

    all_preds.extend(pred)
    all_true.extend(yb.numpy())

print(classification_report(all_true, all_preds))
print("Macro F1:", f1_score(all_true, all_preds, average="macro"))

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    296598
           1       0.58      0.87      0.70       215
           2       1.00      1.00      1.00     19202
           3       0.98      0.98      0.98      1543
           4       1.00      0.99      0.99     25927
           5       0.88      0.99      0.93       784
           6       0.98      0.98      0.98       808
           7       0.99      0.99      0.99       890
           8       1.00      1.00      1.00         1
           9       1.00      0.20      0.33         5
          10       1.00      0.88      0.93       294
          11       0.99      0.90      0.94       483
          12       0.68      0.93      0.79       221
          13       0.00      0.00      0.00         3
          14       0.00      0.00      0.00        98

    accuracy                           1.00    347072
   macro avg       0.81      0.78      0.77    347072
weighted avg       1.00   

Cell 19.1: Prediction Collapse Check

In [ ]:
import numpy as np
print("Pred class counts:", np.bincount(np.array(all_preds), minlength=len(le.classes_))[:20])
print("True class counts:", np.bincount(np.array(all_true), minlength=len(le.classes_))[:20])


Pred class counts: [296591    319  19214   1543  25823    882    802    894      1      1
    258    443    301      0      0]
True class counts: [296598    215  19202   1543  25927    784    808    890      1      5
    294    483    221      3     98]


In [ ]:
from data_pipeline import get_full_pipeline


(loaders, scaler, le) = get_full_pipeline()
train_loader, val_loader, test_loader, n_features, n_classes = loaders

print(n_features, n_classes)

ModuleNotFoundError: No module named 'data_pipeline'